In [ ]:
!pip install datasets sentence-transformers pyarrow

In [ ]:
import os
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from google.colab import drive

# --- 1. Mount Google Drive ---
# This allows us to save progress permanently.
# If Colab disconnects, you don't lose the files you already saved.
drive.mount('/content/drive')

# Create a folder in your drive to store the chunks
output_dir = '/content/drive/My Drive/clinical_trials_embeddings'
os.makedirs(output_dir, exist_ok=True)

# --- 2. Load Model ---
print("Loading model...")
model = SentenceTransformer(
  "thomas-sounack/BioClinical-ModernBERT-base",
  device="cuda"
)

# --- 3. Load the Full Dataset ---
# We load the whole thing, but we will process it in pieces.
ds = load_dataset("louisbrulenaudet/clinical-trials", split="train")
total_rows = len(ds)
print(f"Total rows to process: {total_rows}")

# --- 4. Define Parameters ---
columns_to_embed = ["brief_summary", "eligibility_criteria"]
BATCH_SIZE = 256  # How many rows the GPU processes at once
SHARD_SIZE = 20000 # How many rows we save to disk at a time (save every 20k rows)

def embed_texts(batch):
    for col in columns_to_embed:
        # Check if the text is None (empty), replace with empty string to prevent errors
        texts = [t if t is not None else "" for t in batch[col]]
        embeddings = model.encode(texts, show_progress_bar=False)
        batch[f"{col}_embedding"] = embeddings
    return batch

# --- 5. Process in Shards (The "Save as you go" method) ---
num_shards = (total_rows // SHARD_SIZE) + 1

for shard_idx in range(num_shards):
    # Define filename for this chunk
    filename = os.path.join(output_dir, f"shard_{shard_idx}.parquet")

    # Check if this file already exists in Drive
    if os.path.exists(filename):
        print(f"Shard {shard_idx} already exists. Skipping...")
        continue # Skip to next shard

    print(f"Processing shard {shard_idx + 1}/{num_shards}...")

    # Calculate start and end indices for this shard
    start = shard_idx * SHARD_SIZE
    end = min((shard_idx + 1) * SHARD_SIZE, total_rows)

    # Slice the dataset (this creates a smaller dataset object)
    ds_shard = ds.select(range(start, end))

    # Apply embeddings
    ds_shard_embedded = ds_shard.map(
        embed_texts,
        batched=True,
        batch_size=BATCH_SIZE,
        desc=f"Embedding shard {shard_idx}"
    )

    # Convert ONLY this shard to pandas and save
    df_shard = ds_shard_embedded.to_pandas()
    df_shard.to_parquet(filename)

    print(f"Saved {filename}")

print("All shards processed!")

Mounted at /content/drive
Loading model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Saved /content/drive/My Drive/clinical_trials_embeddings/shard_25.parquet
Processing shard 27/28...


Embedding shard 26:   0%|          | 0/20000 [00:00<?, ? examples/s]

Saved /content/drive/My Drive/clinical_trials_embeddings/shard_26.parquet
Processing shard 28/28...


Embedding shard 27:   0%|          | 0/1897 [00:00<?, ? examples/s]

Saved /content/drive/My Drive/clinical_trials_embeddings/shard_27.parquet
All shards processed!
